In [76]:
# Import packages
import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold

In [77]:
# Load in dataset
path = Path("../../../../local3/sswee/music_download/physionet.org/files/music-sudden-cardiac-death/1.0.1/subject-info.csv")

df = pd.read_csv(
    path,
    sep=";",          # correct delimiter
    decimal=",",      # European decimal format
    engine="python",  # handle irregular formatting
    na_values=["", "NA"] # handles missing values
)

# Clean (tabs inside numbers)
df = df.replace(r"\t", ".", regex=True)

# Convert age to numeric
df["Age"] = (
    df["Age"]
    .astype(str)
    .str.strip()
    .str.replace(",", ".", regex=False)
)
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

print(df.shape) # 992 patients with 103 columns

df.head()

(992, 103)


,Patient ID,Follow-up period from enrollment (days),days_4years,Exit of the study,Cause of death,Age,Gender (male=1),Weight (kg),Height (cm),Body Mass Index (Kg/m2),...,Angiotensin-II receptor blocker (yes=1),Anticoagulants/antitrombotics (yes=1),Betablockers (yes=1),Digoxin (yes=1),Loop diuretics (yes=1),Spironolactone (yes=1),Statins (yes=1),Hidralazina (yes=1),ACE inhibitor (yes=1),Nitrovasodilator (yes=1)
0,P0001,2065,1460,NaN,0,58.0,1,83,163,31.2,...,0,1,1,1,1,0,0,0,1,0
1,P0002,2045,1460,NaN,0,58.0,1,74,160,28.9,...,1,1,1,0,0,0,1,0,0,0
2,P0003,2044,1460,NaN,0,69.0,1,83,174,27.4,...,1,1,1,1,1,0,0,0,0,0
3,P0004,2044,1460,NaN,0,56.0,0,84,165,30.9,...,1,1,1,0,1,1,0,0,0,0
4,P0005,2043,1460,NaN,0,70.0,1,97,183,29.0,...,0,1,1,0,1,0,1,0,1,1


In [80]:
# Patient selection
mask = (
    (df["Holter available"] != 0) &  # Select patients with Holter
    (df["Cause of death"].isin([0, 3, 6, 7])) &  # Known survival or cardiac death
    (df["Prior implantable device"] == 0) &  # Remove pacemaker patients
    ((df["Exit of the study"].ne(2)) | (df["Exit of the study"].isna()))  # Remove cardiac transplant
)

df2 = df[mask].copy()

# Combine Cause of death codes:
# 6 and 7 -> Pump failure death
df2.loc[df2["Cause of death"].isin([6, 7]), "Cause of death"] = 6

# Table of number of patients per outcome
cause_counts = df2["Cause of death"].value_counts().sort_index()
print(cause_counts)

# Note: Original paper reports 94/996 SCDs (9.4%) and 111/996 PFDs (11.1%)
# Final counts align very well with 74/746 SCDs (9.9%) and 87/746 PFds (11.7%)

Cause of death
0    585
3     74
6     87
Name: count, dtype: int64


In [90]:
# Select Patient ID and Cause of death columns (labels for training/validating/testing)
labels = df2[["Patient ID", "Cause of death"]].copy()
labels = labels.rename(columns={"Cause of death": "label"})

# Reset index so it matches positional indices
labels = labels.reset_index(drop=True)

labels["label"].value_counts()

# Stratified K-fold
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

labels["fold"] = -1

for fold, (_, val_idx) in enumerate(
    skf.split(labels["Patient ID"], labels["label"])
):
    labels.loc[val_idx, "fold"] = fold

# Check counts in each fold
labels["fold"].value_counts()
labels.groupby(["fold", "label"]).size().unstack()

label,0,3,6
fold,,,
0,117,15,18
1,117,15,17
2,117,15,17
3,117,15,17
4,117,14,18


In [91]:
# Save labels
path = Path("../../../../local3/sswee/music_download/physionet.org/files/music-sudden-cardiac-death/1.0.1")
out_file = path / "music_patient_folds_5cv.csv"
labels.to_csv(out_file, index=False)

In [92]:
# Smoke test (test on 30 patients, 10 from each group)
# Reproducibility
RANDOM_STATE = 42

# Sanity check
assert set(labels["label"].unique()) == {0, 3, 6}

# Sample 10 patients from each class
sampled = (
    labels
    .groupby("label", group_keys=False)
    .apply(lambda x: x.sample(n=10, random_state=RANDOM_STATE))
    .reset_index(drop=True)
)

print(sampled["label"].value_counts())

path = Path("../../../../local3/sswee/music_download/physionet.org/files/music-sudden-cardiac-death/1.0.1")

out_file = path / "music_patient_test30.csv"
sampled.to_csv(out_file, index=False)

print(f"Saved test cohort to: {out_file.resolve()}")


label
0    10
3    10
6    10
Name: count, dtype: int64
Saved test cohort to: /local3/sswee/music_download/physionet.org/files/music-sudden-cardiac-death/1.0.1/music_patient_test30.csv


/tmp/ipykernel_823257/4139209444.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=10, random_state=RANDOM_STATE))
